### Data Preparation, Exploratory Data Analysis (EDA), Feature Importance, and Model Selection steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline

: 

## 1. Data Loading and Basic Cleaning
First, we load the dataset and check for missing values, basic statistics, and ranges.

In [ ]:
# Ensure you have downloaded insurance.csv from Kaggle to the same directory
df = pd.read_csv('insurance.csv')
df.head()

In [ ]:
# Check for missing values and data types
print(df.isnull().sum())
print("\n")
df.info()

In [ ]:
# Look at the min, max, and ranges of numerical values
df.describe().round(2)

## 2. Exploratory Data Analysis (EDA)
According to the Zoomcamp criteria, we need to analyze the target variable, numerical/categorical features, and feature importance.

### Target Variable Analysis (`charges`)
The target variable represents the medical costs. Let's see its distribution. Is it normally distributed, or skewed (like most financial/cost data)?

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['charges'], bins=50, kde=True)
plt.title('Distribution of Medical Charges')
plt.xlabel('Charges ($)')
plt.show()

In [ ]:
# The distribution is highly right-skewed. 
# Applying a log transformation can often help linear models perform better.
plt.figure(figsize=(8, 5))
sns.histplot(np.log1p(df['charges']), bins=50, kde=True)
plt.title('Log Distribution of Medical Charges')
plt.xlabel('Log(Charges)')
plt.show()

### Feature Analysis
Let's look at how categorical variables affect the charges.

In [ ]:
categorical_columns = ['sex', 'smoker', 'region']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(categorical_columns):
    sns.boxplot(x=df[col], y=df['charges'], ax=axes[i])
    axes[i].set_title(f'Charges vs {col.capitalize()}')
    
plt.tight_layout()
plt.show()

### Numerical Feature Interactions
How does BMI and Age interact with being a smoker?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df, alpha=0.7)
plt.title('BMI vs Charges, colored by Smoking Status')
plt.show()

## 3. Feature Importance Analysis
For numerical features, we can use correlation. For categorical features, we can use Mutual Information.

In [ ]:
numerical_columns = ['age', 'bmi', 'children', 'charges']
correlation_matrix = df[numerical_columns].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## 4. Setting up Validation Framework
Let's split the dataset into train, validation, and test sets (60% / 20% / 20%).

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1) # 0.25 * 0.8 = 0.2

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = np.log1p(df_train.charges.values)
y_val = np.log1p(df_val.charges.values)
y_test = np.log1p(df_test.charges.values)

del df_train['charges']
del df_val['charges']
del df_test['charges']

print(f"Train size: {len(df_train)}, Val size: {len(df_val)}, Test size: {len(df_test)}")